# TabICL on a small dataset — in the browser (wasm32 real-torch)

[TabICL](https://github.com/soda-inria/tabicl) is a scikit-learn-compatible
tabular in-context-learning classifier built on PyTorch. This notebook runs
`fit` + `predict` on a small synthetic dataset entirely client-side in the
xeus-python **wasm32** kernel, against the reduced CPU-only `torch` built by
this prototype.

Two shims (see `tabicl_wasm_shim.py`) bridge the reduced runtime: a pure-Python
`psutil` stub, and a `torch.from_numpy`/`Tensor.numpy` reimplementation via
`.tolist()` (the build is `USE_NUMPY=0`, so the C++ numpy bridge is absent).

In [ ]:
import tabicl_wasm_shim  # noqa: F401  applies psutil stub + numpy-bridge patch
import sys, platform, torch, sklearn, numpy as np
print('python', sys.version.split()[0], '| platform', platform.system())
print('torch', torch.__version__, '| sklearn', sklearn.__version__, '| numpy', np.__version__)

In [ ]:
# Locate the bundled pretrained checkpoint (110 MB; not fetched at runtime).
import os
CKPT = 'tabicl-classifier-v2-20260212.ckpt'
cands = [CKPT, os.path.join(os.getcwd(), CKPT), '/drive/' + CKPT,
         os.path.join('files', CKPT)]
ckpt_path = next((p for p in cands if os.path.exists(p)), None)
print('checkpoint found at:', ckpt_path)
print('candidates checked:', cands)

In [ ]:
# Build a small 3-class synthetic dataset with plain NumPy. (We avoid
# sklearn.datasets/model_selection here because sklearn.datasets imports
# `requests`, which is not part of this minimal wasm environment.)
import numpy as np
rng = np.random.RandomState(0)
n_per = 80
centers = np.array([[0, 0, 0, 0, 0, 0], [3, 3, 0, 0, 1, -2],
                    [-3, 1, 2, -1, 0, 3]], dtype=float)
X = np.vstack([rng.randn(n_per, 6) + c for c in centers])
y = np.array([0] * n_per + [1] * n_per + [2] * n_per)
perm = rng.permutation(len(y))
X, y = X[perm], y[perm]
X_train, X_test, y_train, y_test = X[:180], X[180:], y[:180], y[180:]
print('train', X_train.shape, 'test', X_test.shape, 'classes', sorted(set(y.tolist())))

In [ ]:
from tabicl import TabICLClassifier
from sklearn.metrics import accuracy_score
# Minimal-compute config for single-threaded wasm: few estimators, no AMP,
# no flash-attn, no disk offload (avoids the memmap/psutil-sized paths).
clf = TabICLClassifier(
    n_estimators=1, device='cpu', use_amp=False, use_fa3=False,
    offload_mode=False, batch_size=8, n_jobs=1, verbose=True,
    model_path=ckpt_path, allow_auto_download=False, random_state=42)
try:
    clf.fit(X_train, y_train)
    print('fit done; classes_', clf.classes_)
    y_hat = clf.predict(X_test)
    acc = accuracy_score(y_test, y_hat)
    print('predictions[:12]', list(map(int, y_hat[:12])))
    print('accuracy', round(float(acc), 3))
    assert acc > 0.6, 'accuracy unexpectedly low'
    print('TABICL SUCCESS: fit+predict ran in wasm; accuracy %.3f' % acc)
except Exception as e:
    import traceback; traceback.print_exc()
    print('TABICL BLOCKER:', type(e).__name__, str(e)[:400])